# 04b · ¿Otras fuentes de datos mejorarían el modelo? Prueba final e importancia de variables

> **Reto 2 del INE:** "¿Qué zonas podrían presentar mayor abstención en el futuro si no se interviene?"

**A qué responde este paso.** El modelo del notebook 04 usa 34 variables. Antes de cerrar queremos saber dos cosas:

1. **¿Nos faltó información?** Probamos 18 variables nuevas de otras fuentes: INEGI (censos completos y mapa de municipios) e INE (casillas). Para cada una escribimos antes la hipótesis de por qué podría ayudar.
2. **¿Qué pesa de verdad en el modelo?** Medimos la importancia de cada variable con **permutación**: cuánto empeora la predicción de una elección que el modelo no vio si "revolvemos" esa variable entre municipios.
3. **¿Otro tipo de modelo predice mejor?** Árbol de decisión, random forest, red neuronal y otros (sección 5).
4. **¿Qué no ve el modelo?** Si su error se agrupa por estado o distrito, los casos atípicos, la tecnología como aproximación a redes sociales, el COVID y el gasto de campaña (secciones 6 a 9).
5. **¿Las variables funcionan también en las elecciones presidenciales?** (sección 10).

**Regla para aceptar una variable nueva (fijada antes de ver resultados):** que vaya con lo que la historia no anticipa en las dos transiciones **y** que, al agregarla al modelo, el error baje en las dos validaciones (aprender de 2015 y predecir 2021, y al revés).

## Resumen de lo que encontramos

1. **12 de las 18 variables nuevas se relacionan de forma estable** con lo que la historia no anticipa. Las más fuertes son la densidad de población, la participación económica de las mujeres, tener radio y los electores por casilla.
2. **Pero casi ninguna mejora la predicción.** Solo tres mejoran en las dos validaciones, y menos de 1 %:
   - la **densidad de población**: el error baja 0.05 y 0.04 puntos (de 5.49 y 6.57);
   - el **promedio de ocupantes por vivienda**: 0.001 y 0.03;
   - el **% de católicos**: 0.05 y 0.003.

   Las demás empeoran al menos una de las dos validaciones. **El modelo ya captura esa información** con las variables que tiene: población, rural, escolaridad y pobreza.
3. **Lo que pesa de verdad**, en este orden:
   - la posición del municipio en la presidencial anterior;
   - si elige ayuntamiento el mismo día;
   - el conjunto de variables sociales y económicas;
   - la violencia y la competencia política casi no pesan.
4. **De 14 modelos, el mejor es un ensamble: el promedio de Huber y random forest.** Baja el error en las dos validaciones, fuera del ruido (−0.17 y −0.41 puntos), y gana con la misma regla que usamos para elegir a Huber. Solos, el árbol de decisión, la red neuronal, gradient boosting, SVR y los vecinos cercanos son peores.
5. **Un 28 % del error del modelo es "del estado" y más del 40 % es "del distrito"**, pero ese efecto **no se repite** de una elección a otra. Es la coyuntura de cada elección (campañas locales, conflictos), y no se puede aprender del pasado.
6. **El gasto de campaña (diputaciones federales) no explica ese error.** El COVID tampoco, una vez que se considera lo urbano. La tecnología en la vivienda dice lo mismo que internet.
7. **Encontramos y corregimos un error en los datos:** 7 municipios fuera de Oaxaca eligen por sistemas normativos indígenas y estaban marcados como si eligieran ayuntamiento por partidos (sección 7). La corrección ya está en los notebooks 03 a 05.
8. **En las elecciones presidenciales**, solo elegir ayuntamiento el mismo día y usos y costumbres funcionan igual que en las intermedias. Otras diez variables funcionan con signo opuesto: miden qué tan sensible es el municipio al ciclo presidencial–intermedia (sección 10).
9. **Decisiones de Sofía Ordaz (20 de octubre de 2026):** no agregar variables (D20) y cambiar al ensamble Huber + random forest (D21). Ver la sección 11.

## Hipótesis, escritas antes de probar

| # | Fuente | Variable | Por qué podría ayudar |
|---|---|---|---|
| H1 | INEGI, censos 2010 y 2020 (ITER) | % sin religión, % católica | Las redes comunitarias, entre ellas las religiosas, movilizan el voto |
| H2 | ITER | Desocupación; participación económica, total y de mujeres | El empleo conecta con la vida pública; el desempleo aleja |
| H3 | ITER | % de hogares con jefatura femenina | Señal de migración masculina; el sentido es incierto |
| H4 | ITER | % de casados y de solteros (12 años o más) | Arraigo |
| H5 | ITER | % con discapacidad o limitación | Barreras para llegar a la casilla |
| H6 | ITER | % de viviendas con automóvil, radio, TV o teléfono fijo | Movilidad y acceso a información de campaña |
| H7 | ITER | % de analfabetismo | Barrera para informarse |
| H8 | ITER | % de viviendas deshabitadas | Abandono y migración: lista con gente que ya no vive ahí |
| H9 | ITER | Promedio de ocupantes por vivienda | Hacinamiento; el sentido es incierto |
| H10 | INE, actas | Electores por casilla | Casillas más llenas: filas y distancia |
| H11 | INEGI, mapa de municipios | Densidad de población | Aislamiento frente a la ciudad |

**Sin información del futuro:** para 2015 usamos el censo 2010 y las casillas de 2012; para 2021, el censo 2020 y las casillas de 2018; para 2027, el censo 2020 y las casillas de 2024. La densidad usa el área de los municipios de 2024 (los límites casi no cambian).

**Advertencia:** la pregunta de discapacidad cambió entre censos ("limitación" en 2010, "discapacidad" en 2020). Como cada transición usa su propio censo, la comparación es dentro de cada año.

In [1]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'   # también en los procesos en paralelo de scikit-learn
from pathlib import Path
import csv
import hashlib
import io
import warnings
import zipfile

import geopandas as gpd
import numpy as np
import polars as pl
from scipy.stats import rankdata, spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_tbl_width_chars(200)

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() and (p / 'data').is_dir())
P = raiz / 'data/processed'
ITER = {2010: ('data/raw/externos/inegi/2010/iter_nal_2010_csv.zip', 'iter_00_cpv2010/conjunto_de_datos/iter_00_cpv2010.csv'),
        2020: ('data/raw/externos/inegi/2020/iter_00_cpv2020_csv.zip', 'iter_00_cpv2020/conjunto_de_datos/conjunto_de_datos_iter_00CSV20.csv')}
with (raiz / 'data/reference/huellas_fuentes_externas.csv').open(encoding='utf-8') as f:
    huellas = {r['archivo_relativo']: r['sha256'] for r in csv.DictReader(f)}
for z, _ in ITER.values():
    with (raiz / z).open('rb') as f:
        assert hashlib.file_digest(f, 'sha256').hexdigest() == huellas[z], f'{z} no coincide con su huella'
assert (P / 'variables_transiciones.parquet').exists(), 'Primero corre el notebook 03b'
variables = pl.read_parquet(P / 'variables_transiciones.parquet')
rng = np.random.default_rng(2027)
print('Fuentes verificadas contra sus huellas.')

Fuentes verificadas contra sus huellas.


## 1 · Construimos las variables nuevas

In [2]:
def iter_municipal(anio):
    z, miembro = ITER[anio]
    d = pl.read_csv(io.BytesIO(zipfile.ZipFile(raiz / z).read(miembro)), infer_schema=False, encoding='utf8-lossy')
    d = d.rename({c: c.lower().replace('﻿', '') for c in d.columns}).filter((pl.col('loc') == '0000') & (pl.col('mun') != '000'))
    n = lambda c: pl.col(c).cast(pl.Float64, strict=False)
    return d.select(
        cvegeo=pl.col('entidad') + pl.col('mun'),
        pct_sin_religion=n('psin_relig') / n('pobtot') * 100,
        pct_catolica=n('pcatolica') / n('pobtot') * 100,
        pct_desocupacion=n('pdesocup') / n('pea') * 100,
        tasa_participacion_economica=n('pea') / n('p_12ymas') * 100,
        tasa_participacion_economica_mujeres=n('pea_f') / n('p_12ymas_f') * 100,
        pct_hogares_jefatura_femenina=n('hogjef_f') / n('tothog') * 100,
        pct_casados_12mas=n('p12ym_casa') / n('p_12ymas') * 100,
        pct_solteros_12mas=n('p12ym_solt') / n('p_12ymas') * 100,
        pct_discapacidad_limitacion=n('pcon_lim' if anio == 2010 else 'pcon_disc') / n('pobtot') * 100,
        pct_viviendas_automovil=n('vph_autom') / n('tvivparhab') * 100,
        pct_viviendas_radio=n('vph_radio') / n('tvivparhab') * 100,
        pct_viviendas_tv=n('vph_tv') / n('tvivparhab') * 100,
        pct_viviendas_telefono_fijo=n('vph_telef') / n('tvivparhab') * 100,
        pct_analfabetismo=n('p15ym_an') / n('p_15ymas') * 100,
        pct_viviendas_deshabitadas=n('vivpar_des') / n('tvivpar') * 100,
        promedio_ocupantes_vivienda=n('prom_ocup'),
        poblacion_censo=n('pobtot'))
censos = {a: iter_municipal(a) for a in ITER}

secciones = pl.read_parquet(P / 'secciones_territorio_2024.parquet')
def electores_por_casilla(anio):
    lista = (pl.scan_parquet(P / f'ccpc/ccpc-{anio}.parquet').group_by(pl.col('EDOCVE').cast(pl.Int64), 'SECCION').agg(pl.col('LN').sum()).collect())
    s = secciones.filter(pl.col('AELEC') == anio, pl.col('hay_actas').fill_null(False)).with_columns(pl.col('EDOCVE').cast(pl.Int64)).join(lista, on=['EDOCVE', 'SECCION'])
    return s.group_by(pl.col('municipio_2024').alias('cvegeo')).agg(electores_por_casilla=pl.col('LN').sum() / pl.col('actas_casillas').sum())
casillas = {a: electores_por_casilla(a) for a in (2012, 2018, 2024)}

mapa = gpd.read_file(P / 'municipios_inegi.geojson')
albers = '+proj=aea +lat_1=14.5 +lat_2=32.5 +lat_0=24 +lon_0=-105 +datum=WGS84 +units=m'   # proyección de área igual para México
area = pl.DataFrame({'cvegeo': mapa['CVEGEO'].astype(str).tolist(), 'km2': (mapa.to_crs(albers).geometry.area / 1e6).tolist()})

CENSO, ACTAS = {2015: 2010, 2021: 2020, 2027: 2020}, {2015: 2012, 2021: 2018, 2027: 2024}
NUEVAS = [c for c in censos[2010].columns if c not in ('cvegeo', 'poblacion_censo')] + ['electores_por_casilla', 'densidad_log']
ampliada = pl.concat([variables.filter(pl.col('eleccion_objetivo') == T).join(censos[CENSO[T]], on='cvegeo', how='left')
                      .join(casillas[ACTAS[T]], on='cvegeo', how='left').join(area, on='cvegeo', how='left')
                      .with_columns(densidad_log=(pl.col('poblacion_censo') / pl.col('km2')).log()) for T in (2015, 2021, 2027)], how='diagonal_relaxed')
no_finitos = {c: ampliada.select((~pl.col(c).is_finite()).sum()).item() for c in NUEVAS}
ampliada = ampliada.with_columns([pl.when(pl.col(c).is_finite()).then(pl.col(c)).alias(c) for c in NUEVAS])
print('Valores no finitos que pasan a vacío:', {k: v for k, v in no_finitos.items() if v})
print(f'Área total de los municipios: {area["km2"].sum():,.0f} km² (México: unos 1.96 millones)')
ampliada.group_by('eleccion_objetivo').agg(pl.len().alias('Municipios'), *[pl.col(c).null_count().alias(f'vacíos: {c}') for c in ['pct_sin_religion', 'electores_por_casilla', 'densidad_log']]).sort('eleccion_objetivo')

Valores no finitos que pasan a vacío: {'electores_por_casilla': 3}
Área total de los municipios: 1,962,292 km² (México: unos 1.96 millones)


eleccion_objetivo,Municipios,vacíos: pct_sin_religion,vacíos: electores_por_casilla,vacíos: densidad_log
i16,u32,u32,u32,u32
2015,2477,21,4,21
2021,2477,8,0,8
2027,2477,8,2,8


Los municipios sin dato del censo son los que se crearon después (21 en el censo 2010, 8 en el de 2020). El modelo los completa con la mediana, igual que en el notebook 04.

## 2 · ¿Van con lo que la historia no anticipa?

Igual que en el notebook 03b (sección 5): correlación de Spearman con el residuo de la intermedia dado la presidencial anterior, con intervalo de confianza de 95 % por bootstrap. "Estable" = mismo signo y distinta de cero en las dos transiciones.

In [3]:
def con_resultado(T):
    return ampliada.filter((pl.col('eleccion_objetivo') == T) & pl.col('desviacion_objetivo').is_not_null()
                           & ~pl.col('sin_casilla_objetivo').fill_null(False) & pl.col('desviacion_presidencial').is_not_null())
def spearman_ic(x, y, veces=400):
    ok = ~(np.isnan(x) | np.isnan(y)); x, y = x[ok], y[ok]
    bs = [spearmanr(x[i], y[i]).statistic for i in (rng.integers(0, len(x), len(x)) for _ in range(veces))]
    return spearmanr(x, y).statistic, np.percentile(bs, 2.5), np.percentile(bs, 97.5)
filas = []
for T in (2015, 2021):
    d = con_resultado(T)
    pendiente, ordenada = np.polyfit(d['desviacion_presidencial'], d['desviacion_objetivo'], 1)
    residuo = d['desviacion_objetivo'].to_numpy() - (ordenada + pendiente * d['desviacion_presidencial'].to_numpy())
    for v in NUEVAS:
        r, bajo, alto = spearman_ic(d[v].cast(pl.Float64).to_numpy(), residuo)
        filas.append({'variable': v, 'T': T, 'txt': f'{r:.2f} [{bajo:.2f}, {alto:.2f}]', 'bajo': bajo, 'alto': alto, 'r': r})
estabilidad = (pl.DataFrame(filas).group_by('variable').agg(pl.col('txt').sort_by('T'), pl.col('r').sort_by('T'),
               ((pl.col('bajo') > 0).all() | (pl.col('alto') < 0).all()).alias('estable'))
               .with_columns(pl.col('txt').list.get(0).alias('2012 → 2015: ρ [IC 95 %]'), pl.col('txt').list.get(1).alias('2018 → 2021: ρ [IC 95 %]'),
                             pl.col('r').list.mean().abs().alias('fuerza'))
               .sort('estable', 'fuerza', descending=True))
estabilidad.select('variable', '2012 → 2015: ρ [IC 95 %]', '2018 → 2021: ρ [IC 95 %]', pl.col('estable').replace_strict({True: 'sí', False: 'no'}).alias('¿Estable en las dos?'))

variable,2012 → 2015: ρ [IC 95 %],2018 → 2021: ρ [IC 95 %],¿Estable en las dos?
str,str,str,str
"""densidad_log""","""-0.30 [-0.34, -0.27]""","""-0.19 [-0.22, -0.15]""","""sí"""
"""tasa_participacion_economica_mujeres""","""-0.19 [-0.23, -0.16]""","""-0.18 [-0.22, -0.14]""","""sí"""
"""pct_viviendas_radio""","""-0.21 [-0.25, -0.18]""","""-0.13 [-0.17, -0.09]""","""sí"""
"""tasa_participacion_economica""","""-0.20 [-0.24, -0.15]""","""-0.13 [-0.16, -0.09]""","""sí"""
"""electores_por_casilla""","""-0.18 [-0.22, -0.14]""","""-0.14 [-0.17, -0.10]""","""sí"""
"""pct_catolica""","""0.22 [0.19, 0.26]""","""0.08 [0.04, 0.12]""","""sí"""
"""pct_casados_12mas""","""0.08 [0.04, 0.12]""","""0.19 [0.15, 0.23]""","""sí"""
"""pct_analfabetismo""","""0.15 [0.11, 0.19]""","""0.12 [0.08, 0.15]""","""sí"""
"""pct_sin_religion""","""-0.16 [-0.20, -0.12]""","""-0.10 [-0.14, -0.06]""","""sí"""


**Qué vemos:**

- **12 de las 18 son estables.** Las más fuertes:
  - **densidad** (−0.30 y −0.19): los municipios más densos votan menos de lo que su historia anticipa;
  - **participación económica de las mujeres** (−0.19 y −0.18);
  - **radio** (−0.21 y −0.13);
  - **electores por casilla** (−0.18 y −0.14).
- **Algunos signos contradicen la intuición.** El analfabetismo (+0.15 y +0.12) y la discapacidad van con votar **más** de lo esperado. No es que esas condiciones ayuden a votar: son rasgos de municipios rurales chicos que, a igualdad de historia, votan más en la intermedia. Es el mismo patrón que la pobreza fuera de Oaxaca (notebook 03b, sección 7).
- **No estables:** solteros, automóvil, ocupantes por vivienda, teléfono fijo, desocupación y TV.

## 3 · ¿Mejoran la predicción?

Que una variable se relacione con algo no basta. Debe **agregar información** que el modelo no tiene. Agregamos cada variable (y luego todas) al modelo Huber del notebook 04, el que estaba vigente al hacer esta prueba, y repetimos las dos validaciones. (Después se adoptó el ensamble Huber + random forest, D21; ver la sección 5.)

In [4]:
HISTORIA = ['desviacion_presidencial', 'desviacion_intermedia_previa', 'vecinos_desviacion_presidencial']
LOCALES = ['gobernador_mismo_dia_intermedia', 'diputacion_local_mismo_dia_intermedia', 'ayuntamiento_mismo_dia_intermedia',
           'gobernador_mismo_dia_presidencial', 'diputacion_local_mismo_dia_presidencial', 'ayuntamiento_mismo_dia_presidencial', 'es_sni']
POLITICAS = ['margen_presidencial', 'partidos_efectivos_presidencial']
LISTA = ['pct_rural', 'pct_hombres', 'pct_20_29', 'pct_65_mas', 'crecimiento_lista', 'cobertura_lista_20mas']
SOCIALES = ['escolaridad_promedio', 'pct_habla_lengua_indigena', 'pct_viviendas_internet', 'pct_residia_otro_pais_hace5', 'pct_residia_otra_entidad_hace5',
            'poblacion_log', 'pct_pobreza', 'pct_pobreza_extrema', 'pct_carencia_salud', 'pct_rezago_educativo', 'marginacion', 'pct_localidades_menos_5000',
            'pct_viviendas_emigrantes', 'pct_viviendas_remesas', 'pct_viviendas_retorno']
VIOLENCIA = ['homicidios_100mil_log']
TODAS = HISTORIA + LOCALES + POLITICAS + LISTA + SOCIALES + VIOLENCIA   # las 34 del notebook 04, en el mismo orden de bloques

def huber(): return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), HuberRegressor(alpha=1.0, epsilon=1.35, max_iter=3000))
def boosting(): return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=1.0, random_state=0)
def percentil(x): return (rankdata(x) - 0.5) / len(x) * 100
def error_centrado(y, p): return np.abs((y - np.median(y)) - (p - np.median(p)))
def acierto_20(y, p):
    k = len(y) // 5
    return len(set(np.argsort(y)[:k]) & set(np.argsort(p)[:k])) / k
A15, A21 = con_resultado(2015), con_resultado(2021)
SENTIDOS = [('aprende de 2015, predice 2021', A15, A21), ('aprende de 2021, predice 2015', A21, A15)]
def predecir(modelo, cols, A, B):
    return modelo().fit(A.select(cols).to_numpy().astype(float), A['desviacion_objetivo'].to_numpy()).predict(B.select(cols).to_numpy().astype(float))
base = {n: predecir(huber, TODAS, A, B) for n, A, B in SENTIDOS}

def comparar(etiqueta, modelo, cols, veces=1000):
    fila = {'Modelo': etiqueta}
    for n, A, B in SENTIDOS:
        y = B['desviacion_objetivo'].to_numpy(); p = predecir(modelo, cols, A, B)
        dif = error_centrado(y, p) - error_centrado(y, base[n])
        bs = [dif[i].mean() for i in (rng.integers(0, len(dif), len(dif)) for _ in range(veces))]
        corto = '15→21' if n.startswith('aprende de 2015') else '21→15'
        fila[f'Error {corto} (pp)'] = round(error_centrado(y, p).mean(), 2)
        fila[f'Cambio {corto} [IC 95 %]'] = f'{dif.mean():+.3f} [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]'
        fila[f'Acierto 20 % {corto}'] = round(acierto_20(y, p) * 100, 1)
        fila[f'_mejora_{corto}'] = bool(np.percentile(bs, 97.5) < 0)
    return fila

filas = [comparar('Huber, 34 variables (referencia)', huber, TODAS)]
filas += [comparar(f'+ {v}', huber, TODAS + [v]) for v in NUEVAS]
filas.append(comparar(f'+ las {estabilidad["estable"].sum()} estables', huber, TODAS + estabilidad.filter(pl.col('estable'))['variable'].to_list()))
filas.append(comparar('+ las 18 nuevas', huber, TODAS + NUEVAS))
filas.append(comparar('Gradient boosting, 34 variables', boosting, TODAS))
filas.append(comparar('Gradient boosting + las 18 nuevas', boosting, TODAS + NUEVAS))
resultado = pl.DataFrame(filas)
mejoran_en_las_dos = resultado.filter(pl.col('_mejora_15→21') & pl.col('_mejora_21→15'))['Modelo'].to_list()
print('Mejoran el error, fuera del ruido, en las dos validaciones:', mejoran_en_las_dos)
resultado.drop('_mejora_15→21', '_mejora_21→15')

Mejoran el error, fuera del ruido, en las dos validaciones: ['+ pct_catolica', '+ promedio_ocupantes_vivienda', '+ densidad_log']


Modelo,Error 15→21 (pp),Cambio 15→21 [IC 95 %],Acierto 20 % 15→21,Error 21→15 (pp),Cambio 21→15 [IC 95 %],Acierto 20 % 21→15
str,f64,str,f64,f64,str,f64
"""Huber, 34 variables (referencia)""",5.49,"""+0.000 [+0.000, +0.000]""",71.5,6.57,"""+0.000 [+0.000, +0.000]""",58.2
"""+ pct_sin_religion""",5.43,"""-0.060 [-0.071, -0.049]""",72.3,6.6,"""+0.037 [+0.029, +0.046]""",57.6
"""+ pct_catolica""",5.44,"""-0.051 [-0.065, -0.036]""",71.7,6.56,"""-0.003 [-0.005, -0.002]""",58.4
"""+ pct_desocupacion""",5.49,"""+0.005 [+0.002, +0.008]""",71.5,6.6,"""+0.035 [+0.022, +0.050]""",58.0
"""+ tasa_participacion_economica""",5.51,"""+0.028 [+0.001, +0.054]""",71.0,6.58,"""+0.016 [+0.011, +0.020]""",58.2
"""+ tasa_participacion_economica_mujeres""",5.52,"""+0.037 [+0.000, +0.078]""",71.7,6.57,"""+0.005 [+0.003, +0.007]""",58.2
"""+ pct_hogares_jefatura_femenina""",5.53,"""+0.043 [+0.007, +0.077]""",71.7,6.54,"""-0.028 [-0.032, -0.023]""",57.8
"""+ pct_casados_12mas""",5.56,"""+0.075 [+0.055, +0.095]""",71.0,6.55,"""-0.016 [-0.022, -0.012]""",58.2
"""+ pct_solteros_12mas""",5.54,"""+0.056 [+0.035, +0.079]""",70.4,6.57,"""-0.001 [-0.003, +0.000]""",58.4


**Cómo leer la tabla:** "Cambio" es la diferencia de error contra Huber con las 34 variables, en puntos. **Negativo = mejora.** El intervalo viene de un bootstrap pareado sobre los municipios de la elección probada: si no cruza el cero, la diferencia no es ruido.

**Qué vemos:**

- **Solo tres variables mejoran en las dos validaciones, y muy poco:**
  - la densidad: el error baja 0.05 pp al predecir 2021 y 0.04 pp al predecir 2015, menos del 1 %;
  - los ocupantes por vivienda: 0.001 y 0.03 pp;
  - el % de católicos: 0.05 y 0.003 pp.

  El acierto en el 20 % de mayor riesgo casi no cambia.
- **Las demás empeoran al menos una validación.** Algunas, como el automóvil o el % sin religión, mejoran 2021 pero empeoran 2015: se acomodan a una elección, no siguen un patrón. Otras, como la participación económica o la desocupación, empeoran o no cambian en las dos.
- **Con las 18 juntas:** al predecir 2021 la diferencia está dentro del ruido; al predecir 2015 mejora 0.10 pp. No pasa la regla.
- **Gradient boosting es mucho peor:** 7.3 puntos de error al predecir 2021 contra 5.5, y su acierto en el 20 % de mayor riesgo cae de 72 % a 42 %. Más variables no lo arreglan. Con dos elecciones para aprender, un modelo flexible se aprende de memoria la elección de entrenamiento.

**Por qué variables estables no mejoran la predicción:** dicen lo mismo que variables que el modelo ya tiene. Por ejemplo, la densidad casi repite población y lo rural, y el radio repite marginación e internet. El modelo ya tenía esa información.

## 4 · ¿Qué pesa de verdad? Importancia por permutación

Para cada variable, y luego para cada bloque, "revolvemos" sus valores entre municipios en la elección que el modelo **no vio** y medimos cuánto sube el error. Si sube mucho, el modelo depende de ella. Se repite 30 veces y se promedia. Se mide sobre Huber, la parte lineal del modelo.

Los bloques son más confiables que las variables sueltas: cuando dos variables dicen lo mismo, revolver una sola casi no daña, porque la otra la sustituye.

In [5]:
def importancia(A, B, grupos, veces=30):
    m = huber().fit(A.select(TODAS).to_numpy().astype(float), A['desviacion_objetivo'].to_numpy())
    X, y = B.select(TODAS).to_numpy().astype(float), B['desviacion_objetivo'].to_numpy()
    e0 = error_centrado(y, m.predict(X)).mean()
    salida = {}
    for g, vs in grupos.items():
        idx = [TODAS.index(v) for v in vs]; aumentos = []
        for _ in range(veces):
            Xp = X.copy(); Xp[:, idx] = X[rng.permutation(len(X))][:, idx]
            aumentos.append(error_centrado(y, m.predict(Xp)).mean() - e0)
        salida[g] = np.mean(aumentos)
    return salida
BLOQUES = {'Historia del municipio (3)': HISTORIA, 'Elecciones locales el mismo día y usos y costumbres (7)': LOCALES,
           'Sociales y económicas (15)': SOCIALES, 'Lista nominal (6)': LISTA, 'Competencia política (2)': POLITICAS, 'Violencia (1)': VIOLENCIA}
imp_bloques = {n: importancia(A, B, BLOQUES) for n, A, B in SENTIDOS}
pl.DataFrame({'Bloque': list(BLOQUES),
              'Aumento del error al revolverlo, prediciendo 2021 (pp)': [round(imp_bloques[SENTIDOS[0][0]][b], 2) for b in BLOQUES],
              'Ídem, prediciendo 2015 (pp)': [round(imp_bloques[SENTIDOS[1][0]][b], 2) for b in BLOQUES]})

Bloque,"Aumento del error al revolverlo, prediciendo 2021 (pp)","Ídem, prediciendo 2015 (pp)"
str,f64,f64
"""Historia del municipio (3)""",7.0,5.61
"""Elecciones locales el mismo día y usos y costumbres (7)""",2.87,3.89
"""Sociales y económicas (15)""",1.84,1.52
"""Lista nominal (6)""",0.07,0.2
"""Competencia política (2)""",0.01,0.04
"""Violencia (1)""",-0.0,-0.01


In [6]:
imp_variables = {n: importancia(A, B, {v: [v] for v in TODAS}) for n, A, B in SENTIDOS}
(pl.DataFrame({'Variable': TODAS, 'Prediciendo 2021 (pp)': [imp_variables[SENTIDOS[0][0]][v] for v in TODAS],
               'Prediciendo 2015 (pp)': [imp_variables[SENTIDOS[1][0]][v] for v in TODAS]})
 .with_columns(pl.exclude('Variable').round(2)).sort('Prediciendo 2021 (pp)', descending=True).head(12))

Variable,Prediciendo 2021 (pp),Prediciendo 2015 (pp)
str,f64,f64
"""desviacion_presidencial""",5.35,5.01
"""ayuntamiento_mismo_dia_intermedia""",1.92,7.34
"""gobernador_mismo_dia_presidencial""",0.6,-0.0
"""ayuntamiento_mismo_dia_presidencial""",0.47,-0.01
"""desviacion_intermedia_previa""",0.38,0.17
"""gobernador_mismo_dia_intermedia""",0.32,0.06
"""es_sni""",0.18,0.26
"""diputacion_local_mismo_dia_presidencial""",0.17,-0.03
"""poblacion_log""",0.12,1.09


**Qué vemos:**

- **La historia del municipio es lo que más pesa:** sin ella, el error sube entre 5.6 y 7 pp. Sobre todo pesa su posición en la presidencial anterior.
- **Le siguen las elecciones locales el mismo día y usos y costumbres** (de 2.9 a 3.9 pp). Elegir ayuntamiento ese día es, por sí sola, la segunda variable más importante.
- **Las variables sociales y económicas importan como conjunto** (de 1.5 a 1.8 pp), pero ninguna sola es indispensable, porque se sustituyen entre sí. Por eso agregar más variables de ese tipo no mejora.
- **La competencia política y la violencia casi no pesan** (menos de 0.05 pp), igual que en el notebook 03b.

## 5 · ¿Otro tipo de modelo predice mejor?

Comparamos 11 tipos de modelo y 3 combinaciones (ensambles), con el mismo protocolo: aprender de una transición y predecir la otra. En los modelos que tienen ajustes (profundidad del árbol, número de neuronas, etc.), los ajustes se eligen con validación cruzada **solo dentro de la elección de entrenamiento**. La elección de prueba nunca se ve antes.

| Modelo | Idea |
|---|---|
| Ridge, lasso, elastic net | Lineales con distintos tipos de regularización |
| Árbol de decisión | Reglas del tipo "si elige ayuntamiento y es rural, entonces…" |
| Random forest, extra trees | Cientos de árboles promediados |
| Gradient boosting | Árboles en secuencia, cada uno corrige al anterior |
| Red neuronal (perceptrón multicapa) | Capas de neuronas; capta relaciones no lineales |
| SVR | Máquina de vectores de soporte con núcleo radial |
| Vecinos cercanos | Predice con los municipios más parecidos |
| Ensambles | Promedio de las predicciones de dos o tres modelos |

**Regla para elegir (la misma del notebook 04, fijada antes):** el menor error promedio en las dos validaciones, que además le gane a la regla simple en las dos.

In [7]:
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

cv = KFold(5, shuffle=True, random_state=0)
pre = lambda: [SimpleImputer(strategy='median'), StandardScaler()]
buscar = lambda est, grid: GridSearchCV(est, grid, cv=cv, scoring='neg_mean_absolute_error', n_jobs=1)   # un solo proceso; los bosques usan varios núcleos por dentro
MODELOS = {
    'Huber': huber,
    'Ridge': lambda: make_pipeline(*pre(), RidgeCV(alphas=np.logspace(-2, 4, 30))),
    'Lasso': lambda: make_pipeline(*pre(), LassoCV(cv=cv, alphas=60, max_iter=20000)),
    'Elastic net': lambda: make_pipeline(*pre(), ElasticNetCV(cv=cv, l1_ratio=[.1, .5, .9], alphas=40, max_iter=20000)),
    'Árbol de decisión': lambda: buscar(make_pipeline(SimpleImputer(strategy='median'), DecisionTreeRegressor(random_state=0)),
                                        {'decisiontreeregressor__max_depth': [3, 4, 6, 8], 'decisiontreeregressor__min_samples_leaf': [20, 50, 100]}),
    'Random forest': lambda: buscar(make_pipeline(SimpleImputer(strategy='median'), RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)),
                                    {'randomforestregressor__min_samples_leaf': [5, 20, 50], 'randomforestregressor__max_features': [0.33, 0.6, 1.0]}),
    'Extra trees': lambda: buscar(make_pipeline(SimpleImputer(strategy='median'), ExtraTreesRegressor(n_estimators=300, random_state=0, n_jobs=-1)),
                                  {'extratreesregressor__min_samples_leaf': [5, 20, 50], 'extratreesregressor__max_features': [0.33, 0.6, 1.0]}),
    'Gradient boosting': lambda: buscar(HistGradientBoostingRegressor(random_state=0, max_iter=400),
                                        {'learning_rate': [0.02, 0.05], 'max_depth': [2, 3, None], 'min_samples_leaf': [20, 60]}),
    'Red neuronal': lambda: buscar(make_pipeline(*pre(), MLPRegressor(max_iter=3000, early_stopping=True, random_state=0)),
                                   {'mlpregressor__hidden_layer_sizes': [(16,), (64,), (32, 16)], 'mlpregressor__alpha': [0.01, 1.0, 10.0]}),
    'SVR': lambda: buscar(make_pipeline(*pre(), SVR(kernel='rbf')), {'svr__C': [1, 5, 20], 'svr__epsilon': [0.5, 2.0], 'svr__gamma': ['scale', 0.01]}),
    'Vecinos cercanos': lambda: buscar(make_pipeline(*pre(), KNeighborsRegressor(weights='distance')), {'kneighborsregressor__n_neighbors': [10, 25, 50, 100]}),
}
predicciones, ajustes = {}, {}
for n, A, B in SENTIDOS:
    XA, XB, yA = A.select(TODAS).to_numpy().astype(float), B.select(TODAS).to_numpy().astype(float), A['desviacion_objetivo'].to_numpy()
    predicciones[(n, 'Regla simple')] = B['desviacion_presidencial'].to_numpy() + (A['desviacion_objetivo'] - A['desviacion_presidencial']).median()
    for nombre, f in MODELOS.items():
        m = f().fit(XA, yA)
        predicciones[(n, nombre)] = m.predict(XB)
        if hasattr(m, 'best_params_'):
            ajustes[(n, nombre)] = m.best_params_
    predicciones[(n, 'Ensamble: Huber + random forest')] = (predicciones[(n, 'Huber')] + predicciones[(n, 'Random forest')]) / 2
    predicciones[(n, 'Ensamble: Huber + red neuronal')] = (predicciones[(n, 'Huber')] + predicciones[(n, 'Red neuronal')]) / 2
    predicciones[(n, 'Ensamble: Huber + boosting + red neuronal')] = (predicciones[(n, 'Huber')] + predicciones[(n, 'Gradient boosting')] + predicciones[(n, 'Red neuronal')]) / 3

def metricas_completas(y, p, w):
    k10, k20 = len(y) // 10, len(y) // 5; c = error_centrado(y, p)
    return {'error': c.mean(), 'ponderado': np.average(c, weights=w), 'posicion': np.abs(percentil(y) - percentil(p)).mean(),
            'acierto20': len(set(np.argsort(y)[:k20]) & set(np.argsort(p)[:k20])) / k20 * 100,
            'acierto10': len(set(np.argsort(y)[:k10]) & set(np.argsort(p)[:k10])) / k10 * 100}
filas = []
for (n, nombre), p in predicciones.items():
    B = A21 if n.startswith('aprende de 2015') else A15
    corto = '15→21' if n.startswith('aprende de 2015') else '21→15'
    filas.append({'Modelo': nombre, 'v': corto, **metricas_completas(B['desviacion_objetivo'].to_numpy(), p, B['lista_presidencial'].to_numpy().astype(float))})
zoo = pl.DataFrame(filas).pivot(on='v', index='Modelo', values=['error', 'ponderado', 'posicion', 'acierto20', 'acierto10'])
regla = zoo.filter(pl.col('Modelo') == 'Regla simple').row(0, named=True)
zoo = (zoo.with_columns(error_promedio=(pl.col('error_15→21') + pl.col('error_21→15')) / 2,
                        le_gana_a_la_regla=(pl.col('error_15→21') < regla['error_15→21']) & (pl.col('error_21→15') < regla['error_21→15']))
       .sort('error_promedio'))
GANADOR = zoo.filter(pl.col('le_gana_a_la_regla'))['Modelo'][0]
print(f'Ganador con la regla del notebook 04: {GANADOR}')
zoo.select('Modelo', pl.col('error_promedio').round(2).alias('Error promedio (pp)'),
           pl.col('error_15→21').round(2).alias('Error 15→21'), pl.col('error_21→15').round(2).alias('Error 21→15'),
           pl.col('ponderado_15→21').round(2).alias('Ponderado 15→21'), pl.col('ponderado_21→15').round(2).alias('Ponderado 21→15'),
           pl.col('posicion_15→21').round(1).alias('Posición 15→21'), pl.col('posicion_21→15').round(1).alias('Posición 21→15'),
           pl.col('acierto20_15→21').round(1).alias('Acierto 20 % 15→21'), pl.col('acierto20_21→15').round(1).alias('Acierto 20 % 21→15'),
           pl.col('le_gana_a_la_regla').replace_strict({True: 'sí', False: 'no'}).alias('¿Le gana a la regla simple en las dos?'))

Ganador con la regla del notebook 04: Ensamble: Huber + random forest


Modelo,Error promedio (pp),Error 15→21,Error 21→15,Ponderado 15→21,Ponderado 21→15,Posición 15→21,Posición 21→15,Acierto 20 % 15→21,Acierto 20 % 21→15,¿Le gana a la regla simple en las dos?
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
"""Ensamble: Huber + random forest""",5.74,5.32,6.16,4.3,4.76,10.2,11.9,69.8,59.4,"""sí"""
"""Extra trees""",5.86,5.95,5.76,4.99,5.22,11.3,10.9,62.2,61.1,"""sí"""
"""Random forest""",5.99,5.83,6.15,4.91,5.63,11.2,11.9,62.8,59.8,"""sí"""
"""Huber""",6.03,5.49,6.57,4.68,4.9,10.6,12.7,71.5,58.2,"""sí"""
"""Ensamble: Huber + red neuronal""",6.08,5.58,6.58,4.12,4.87,10.9,13.0,71.0,58.4,"""sí"""
"""Elastic net""",6.1,5.79,6.42,4.76,4.86,11.2,12.6,70.0,58.2,"""sí"""
"""Ridge""",6.12,5.79,6.45,4.79,4.88,11.2,12.6,70.0,58.4,"""sí"""
"""Ensamble: Huber + boosting + red neuronal""",6.13,5.87,6.38,4.2,4.54,11.4,12.6,66.7,57.8,"""sí"""
"""Red neuronal""",6.42,6.05,6.78,4.45,4.97,12.0,13.4,69.4,57.6,"""no"""


In [8]:
# ¿La diferencia del ganador contra Huber es ruido? Bootstrap pareado sobre los municipios de la elección probada
filas = []
for n, A, B in SENTIDOS:
    y = B['desviacion_objetivo'].to_numpy(); e0 = error_centrado(y, predicciones[(n, 'Huber')])
    for nombre in zoo['Modelo'].head(5).to_list():
        if nombre == 'Huber':
            continue
        dif = error_centrado(y, predicciones[(n, nombre)]) - e0
        bs = [dif[i].mean() for i in (rng.integers(0, len(dif), len(dif)) for _ in range(2000))]
        filas.append({'Modelo': nombre, 'Validación': n, 'Cambio contra Huber (pp)': round(dif.mean(), 3),
                      'IC 95 %': f'[{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]'})
print('Ajustes elegidos por validación cruzada interna:')
for k, v in ajustes.items():
    print(' ', k[0], '·', k[1], '·', v)
pl.DataFrame(filas)

Ajustes elegidos por validación cruzada interna:
  aprende de 2015, predice 2021 · Árbol de decisión · {'decisiontreeregressor__max_depth': 8, 'decisiontreeregressor__min_samples_leaf': 20}
  aprende de 2015, predice 2021 · Random forest · {'randomforestregressor__max_features': 0.6, 'randomforestregressor__min_samples_leaf': 5}
  aprende de 2015, predice 2021 · Extra trees · {'extratreesregressor__max_features': 1.0, 'extratreesregressor__min_samples_leaf': 5}
  aprende de 2015, predice 2021 · Gradient boosting · {'learning_rate': 0.02, 'max_depth': None, 'min_samples_leaf': 20}
  aprende de 2015, predice 2021 · Red neuronal · {'mlpregressor__alpha': 10.0, 'mlpregressor__hidden_layer_sizes': (64,)}
  aprende de 2015, predice 2021 · SVR · {'svr__C': 20, 'svr__epsilon': 0.5, 'svr__gamma': 0.01}
  aprende de 2015, predice 2021 · Vecinos cercanos · {'kneighborsregressor__n_neighbors': 10}
  aprende de 2021, predice 2015 · Árbol de decisión · {'decisiontreeregressor__max_depth': 8, 'decisi

Modelo,Validación,Cambio contra Huber (pp),IC 95 %
str,str,f64,str
"""Ensamble: Huber + random forest""","""aprende de 2015, predice 2021""",-0.171,"""[-0.262, -0.077]"""
"""Extra trees""","""aprende de 2015, predice 2021""",0.468,"""[+0.297, +0.645]"""
"""Random forest""","""aprende de 2015, predice 2021""",0.34,"""[+0.176, +0.517]"""
"""Ensamble: Huber + red neuronal""","""aprende de 2015, predice 2021""",0.089,"""[+0.017, +0.159]"""
"""Ensamble: Huber + random forest""","""aprende de 2021, predice 2015""",-0.409,"""[-0.478, -0.341]"""
"""Extra trees""","""aprende de 2021, predice 2015""",-0.806,"""[-0.944, -0.679]"""
"""Random forest""","""aprende de 2021, predice 2015""",-0.414,"""[-0.547, -0.293]"""
"""Ensamble: Huber + red neuronal""","""aprende de 2021, predice 2015""",0.016,"""[-0.039, +0.071]"""


**Qué vemos:**

- **El ganador es el ensamble Huber + random forest** (error promedio de 5.74 contra 6.03 de Huber solo). Gana con la misma regla con la que antes elegimos a Huber.
  - Baja el error al predecir 2021 (5.32 contra 5.49; −0.17 pp, IC de −0.26 a −0.08) y al predecir 2015 (6.16 contra 6.57; −0.41 pp, IC de −0.48 a −0.34).
  - También mejora el error ponderado por personas y el error de posición en las dos validaciones.
  - **Solo pierde en un punto:** el acierto en el 20 % de mayor riesgo al predecir 2021 baja de 71.5 % a 69.8 %, unos 8 municipios de 487.
- **Por qué funciona la combinación:** Huber capta las relaciones generales y es estable. Random forest capta combinaciones que un modelo lineal no ve, por ejemplo "no elige ayuntamiento **y** es rural", pero se acomoda demasiado a la elección con la que aprende (solo, al predecir 2021, se equivoca más que Huber). Al promediarlos, los errores de uno compensan los del otro.
- **Los demás:**
  - el **árbol de decisión** solo es de los peores (9.2 pp al predecir 2021): aprende reglas demasiado rígidas;
  - la **red neuronal** no le gana a la regla simple al predecir 2021 (6.05 contra 6.03): con unos 2,400 municipios por elección hay pocos datos para una red;
  - **gradient boosting, SVR y vecinos cercanos** también pierden contra la regla simple en al menos una validación.
- **Advertencia:** comparamos 14 candidatos con las mismas dos validaciones, así que el ganador puede verse un poco mejor de lo que es. Que mejore **en los dos sentidos y fuera del ruido** reduce ese riesgo, pero no lo elimina.

## 6 · Lo que el modelo no ve: ¿el error se agrupa por estado o por distrito?

Si en un estado hubo una campaña fuerte, un conflicto o una elección local muy competida, todos sus municipios deberían "fallar" en el mismo sentido. Medimos qué parte del error del modelo se explica solo con saber el estado o el distrito federal del municipio. Lo comparamos con lo que se explicaría al azar, revolviendo los estados y distritos.

Después vemos si ese "efecto del estado" **se repite** de una elección a otra. Si se repitiera, se podría aprender del pasado.

In [9]:
secciones_ccpc = secciones
def distrito_principal(anio):
    c = (pl.scan_parquet(P / f'ccpc/ccpc-{anio}.parquet').group_by(pl.col('EDOCVE').cast(pl.Int64), 'SECCION', 'DEF').agg(pl.col('LN').sum()).collect())
    s = secciones_ccpc.filter(pl.col('AELEC') == anio).with_columns(pl.col('EDOCVE').cast(pl.Int64)).join(c, on=['EDOCVE', 'SECCION'])
    return (s.group_by(pl.col('municipio_2024').alias('cvegeo'), 'EDOCVE', 'DEF').agg(pl.col('LN').sum()).sort('LN', descending=True)
            .group_by('cvegeo').first().select('cvegeo', distrito=pl.format('{}-{}', 'EDOCVE', 'DEF')))
def parte_explicada(d, grupo):
    m = d.group_by(grupo).agg(pl.col('error').mean().alias('media'))
    x = d.join(m, on=grupo)
    return 1 - ((x['error'] - x['media']) ** 2).sum() / ((d['error'] - d['error'].mean()) ** 2).sum()
filas, por_estado = [], {}
for n, A, B in SENTIDOS:
    y = B['desviacion_objetivo'].to_numpy(); p = predicciones[(n, 'Huber')]
    d = (B.select('cvegeo', 'clave_estado').with_columns(error=pl.Series((y - np.median(y)) - (p - np.median(p))))
         .join(distrito_principal(2021 if n.startswith('aprende de 2015') else 2015), on='cvegeo', how='left'))
    azar = lambda g: np.mean([parte_explicada(d.with_columns(pl.Series(g, rng.permutation(d[g].to_numpy()))), g) for _ in range(200)])
    filas.append({'Validación': n, 'Explicado por el estado (%)': round(parte_explicada(d, 'clave_estado') * 100, 1), 'Al azar (%)': round(azar('clave_estado') * 100, 1),
                  'Explicado por el distrito (%)': round(parte_explicada(d, 'distrito') * 100, 1), 'Al azar (%) ': round(azar('distrito') * 100, 1),
                  'Distritos': d['distrito'].n_unique()})
    por_estado[n] = d.group_by('clave_estado').agg(pl.col('error').median().alias(n))
print(pl.DataFrame(filas))
repite = por_estado[SENTIDOS[0][0]].join(por_estado[SENTIDOS[1][0]], on='clave_estado')
print(f'¿Se repite el efecto del estado? Spearman entre el error mediano de cada estado al predecir 2021 y al predecir 2015: '
      f'{spearmanr(repite[SENTIDOS[0][0]], repite[SENTIDOS[1][0]]).statistic:.2f} (32 estados)')

shape: (2, 6)
┌───────────────────────────────┬─────────────────────────────┬─────────────┬───────────────────────────────┬──────────────┬───────────┐
│ Validación                    ┆ Explicado por el estado (%) ┆ Al azar (%) ┆ Explicado por el distrito (%) ┆ Al azar (%)  ┆ Distritos │
│ ---                           ┆ ---                         ┆ ---         ┆ ---                           ┆ ---          ┆ ---       │
│ str                           ┆ f64                         ┆ f64         ┆ f64                           ┆ f64          ┆ i64       │
╞═══════════════════════════════╪═════════════════════════════╪═════════════╪═══════════════════════════════╪══════════════╪═══════════╡
│ aprende de 2015, predice 2021 ┆ 27.7                        ┆ 1.3         ┆ 41.4                          ┆ 10.1         ┆ 251       │
│ aprende de 2021, predice 2015 ┆ 27.4                        ┆ 1.3         ┆ 44.4                          ┆ 10.3         ┆ 251       │
└──────────────────────────

**Qué vemos:**

- **El estado explica el 28 % del error del modelo**, contra 1.3 % al azar. **El distrito explica entre 41 % y 44 %**, contra 10 % al azar. Una parte grande de lo que el modelo no acierta es común a todos los municipios de un estado o distrito.
- **Pero no se repite:** los estados donde el modelo se quedó corto en una elección se pasaron en la otra (correlación de −0.46). Es un efecto propio de cada elección: campañas locales, competencia por la gubernatura, conflictos.
- **Consecuencia:** no se puede aprender del pasado. Para 2027, la forma de captarlo es **actualizar el semáforo cuando se conozcan las candidaturas y la coyuntura de cada estado** (primavera de 2027), no agregar más historia.

## 7 · Cambios atípicos

Municipios donde la intermedia de 2021 se separó del modelo más de tres veces el error típico. Son los candidatos para buscar una causa concreta (conflictos, cambios en la forma de elegir autoridades, campañas locales). Aquí solo los listamos; la causa se verifica con fuentes, caso por caso.

In [10]:
y = A21['desviacion_objetivo'].to_numpy(); p = predicciones[(SENTIDOS[0][0], 'Huber')]
error_2021 = (y - np.median(y)) - (p - np.median(p))
tipico = 1.4826 * np.median(np.abs(error_2021 - np.median(error_2021)))
atipicos = (A21.select('estado', 'municipio', 'lista_presidencial', 'desviacion_presidencial', 'desviacion_objetivo')
            .with_columns(prevista=pl.Series(p), error=pl.Series(error_2021)).filter(pl.col('error').abs() > 3 * tipico))
print(f'Error típico: {tipico:.1f} pp · atípicos: {atipicos.height} de {A21.height}')
(atipicos.sort(pl.col('error').abs(), descending=True).head(15)
 .select(pl.col('estado').alias('Estado'), pl.col('municipio').alias('Municipio'), pl.col('lista_presidencial').alias('Lista 2018'),
         pl.col('desviacion_presidencial').round(1).alias('Desviación 2018'), pl.col('prevista').round(1).alias('Prevista 2021'),
         pl.col('desviacion_objetivo').round(1).alias('Real 2021'), pl.col('error').round(1).alias('Error')))

Error típico: 6.3 pp · atípicos: 45 de 2437


Estado,Municipio,Lista 2018,Desviación 2018,Prevista 2021,Real 2021,Error
str,str,i64,f64,f64,f64,f64
"""Puebla""","""San José Miahuatlán""",9606,16.2,27.0,-36.1,-59.3
"""Puebla""","""Teotlalco""",2559,13.9,28.4,-25.6,-50.2
"""Sonora""","""Atil""",841,24.2,39.6,-10.2,-46.0
"""Morelos""","""Hueyapan""",5184,9.2,20.9,-20.5,-37.6
"""Chiapas""","""El Parral""",11035,8.2,18.6,-22.3,-37.2
"""Chiapas""","""Solosuchiapa""",5681,-27.9,-3.7,27.5,35.0
"""Oaxaca""","""Magdalena Tlacotepec""",1083,-2.9,4.2,33.5,33.0
"""Chiapas""","""Pantelhó""",13323,14.0,32.7,-2.1,-30.9
"""Tlaxcala""","""San Juan Huactzinco""",5318,-8.9,2.4,26.6,28.0


**Qué vemos:**

- **45 de 2,437 municipios** se separaron del modelo más de tres veces el error típico (6.3 pp).
- **Esta lista sirvió para encontrar un error en nuestros datos.** En la primera corrida aparecían Ayutla de los Libres, Ñuu Savi y Hueyapan. Al investigarlos confirmamos, con documentos de los institutos electorales y del INE, que **7 municipios fuera de Oaxaca eligen por sistemas normativos indígenas** y no votan ayuntamiento por partidos: Oxchuc, Ayutla de los Libres, Ñuu Savi, Cherán, Coatetelco, Xoxocotla y Hueyapan. Estaban marcados como si eligieran ayuntamiento. Ya se corrigió en el notebook 03 (`data/reference/municipios_sni_fuera_oaxaca.csv`), y por eso subió su riesgo para 2027.
- Algunos siguen en la lista después de la corrección: su caída de 2021 fue mayor de lo que explica no tener ayuntamiento.
- **Para los demás no tenemos una causa verificada.** Por ejemplo, San José Miahuatlán, Teotlalco, Atil y El Parral cayeron mucho en 2021; Solosuchiapa y Magdalena Tlacotepec subieron mucho. Cada caso se tendría que revisar con fuentes, como hicimos en el reporte H.

## 8 · Tecnología (como aproximación a redes sociales) y COVID

**Redes sociales:** no existe un dato público de uso de redes por municipio; la encuesta del INEGI (ENDUTIH) llega a estados y algunas ciudades. Probamos lo más cercano que sí hay por municipio en los censos: viviendas con celular, computadora e internet.

**COVID:** ¿votaron menos en 2021 los municipios con más muertes por COVID? Usamos las defunciones del INEGI con causa U07.1 o U07.2, por municipio de residencia, ocurridas de marzo de 2020 a mayo de 2021. **Solo sirve para explicar 2021, no para predecir 2027.**

In [11]:
E = P / 'externos'
filas = []
for T, censo in [(2015, 2010), (2021, 2020)]:
    tec = pl.read_parquet(E / f'inegi_municipal_{censo}.parquet').select('cvegeo', 'pct_viviendas_con_celular', 'pct_viviendas_con_computadora')
    d = con_resultado(T).join(tec, on='cvegeo', how='left')
    pendiente, ordenada = np.polyfit(d['desviacion_presidencial'], d['desviacion_objetivo'], 1)
    residuo = d['desviacion_objetivo'].to_numpy() - (ordenada + pendiente * d['desviacion_presidencial'].to_numpy())
    for v in ['pct_viviendas_con_celular', 'pct_viviendas_con_computadora', 'pct_viviendas_internet']:
        r, bajo, alto = spearman_ic(d[v].cast(pl.Float64).to_numpy(), residuo)
        filas.append({'Variable': v, 'T': T, 'ρ': f'{r:.2f} [{bajo:.2f}, {alto:.2f}]'})
print(pl.DataFrame(filas).pivot(on='T', index='Variable', values='ρ').rename({'2015': '2012 → 2015', '2021': '2018 → 2021'}))

DEFUNCIONES = {2020: ('data/raw/externos/violencia/inegi/conjunto_de_datos_defunciones_registradas_2020_csv.zip', 'conjunto_de_datos/conjunto_de_datos_defunciones_registrados_2020.csv'),
               2021: ('data/raw/externos/violencia/inegi/conjunto_de_datos_defunciones_registradas_2021_csv.zip', 'conjunto_de_datos/conjunto_de_datos_defunciones_registradas_2021.csv')}
for z, _ in DEFUNCIONES.values():
    with (raiz / z).open('rb') as f:
        assert hashlib.file_digest(f, 'sha256').hexdigest() == huellas[z], f'{z} no coincide con su huella'
defunciones = pl.concat([pl.read_csv(io.BytesIO(zipfile.ZipFile(raiz / z).read(m)), columns=['ent_resid', 'mun_resid', 'causa_def', 'mes_ocurr', 'anio_ocur'], infer_schema=False)
                         for z, m in DEFUNCIONES.values()]).with_columns(pl.col('anio_ocur', 'mes_ocurr').cast(pl.Int32, strict=False))
covid = (defunciones.filter(pl.col('causa_def').is_in(['U071', 'U072']),
                            ((pl.col('anio_ocur') == 2020) & (pl.col('mes_ocurr') >= 3)) | ((pl.col('anio_ocur') == 2021) & (pl.col('mes_ocurr') <= 5)))
         .group_by(cvegeo=pl.col('ent_resid') + pl.col('mun_resid')).agg(muertes_covid=pl.len()))
poblacion_2020 = pl.read_parquet(E / 'conapo_proyecciones_edad_sexo.parquet').filter(pl.col('anio') == 2020).group_by('cvegeo').agg(poblacion=pl.col('poblacion').sum())
d = (A21.join(covid, on='cvegeo', how='left').join(poblacion_2020, on='cvegeo', how='left')
     .with_columns(tasa_covid=pl.col('muertes_covid').fill_null(0) / pl.col('poblacion') * 1e5))
pendiente, ordenada = np.polyfit(d['desviacion_presidencial'], d['desviacion_objetivo'], 1)
residuo = d['desviacion_objetivo'].to_numpy() - (ordenada + pendiente * d['desviacion_presidencial'].to_numpy())
r, bajo, alto = spearman_ic(d['tasa_covid'].to_numpy(), residuo)
r_modelo = spearmanr(d['tasa_covid'].to_numpy(), error_2021, nan_policy='omit').statistic
print(f'Muertes por COVID (mar-2020 a may-2021, registradas en 2020-2021): {covid["muertes_covid"].sum():,} · tasa mediana {d["tasa_covid"].median():.0f} por 100 mil')
print(f'COVID contra lo que 2018 no anticipa: ρ = {r:.2f} [{bajo:.2f}, {alto:.2f}] · contra el error del modelo completo: ρ = {r_modelo:.2f}')

shape: (3, 3)
┌───────────────────────────────┬──────────────────────┬──────────────────────┐
│ Variable                      ┆ 2012 → 2015          ┆ 2018 → 2021          │
│ ---                           ┆ ---                  ┆ ---                  │
│ str                           ┆ str                  ┆ str                  │
╞═══════════════════════════════╪══════════════════════╪══════════════════════╡
│ pct_viviendas_con_celular     ┆ -0.08 [-0.12, -0.04] ┆ -0.07 [-0.11, -0.03] │
│ pct_viviendas_con_computadora ┆ -0.11 [-0.14, -0.07] ┆ -0.11 [-0.15, -0.08] │
│ pct_viviendas_internet        ┆ -0.08 [-0.12, -0.04] ┆ -0.11 [-0.15, -0.07] │
└───────────────────────────────┴──────────────────────┴──────────────────────┘


Muertes por COVID (mar-2020 a may-2021, registradas en 2020-2021): 336,812 · tasa mediana 145 por 100 mil
COVID contra lo que 2018 no anticipa: ρ = -0.12 [-0.16, -0.08] · contra el error del modelo completo: ρ = 0.13


**Qué vemos:**

- **Celular y computadora dicen lo mismo que internet** (de −0.07 a −0.11 en las dos transiciones), que ya está en el modelo. No hay un dato municipal de redes sociales que agregue algo.
- **COVID:** los municipios con más muertes por COVID votaron un poco menos de lo que anticipaba 2018 (−0.12). Pero contra el error del modelo completo la relación se invierte (+0.13). El modelo ya considera lo urbano y el tamaño, que es donde hubo más muertes. **No hay un efecto del COVID aparte de la urbanización.**

## 9 · ¿Las campañas explican lo que el modelo no ve?

No existe un registro público de qué tan intensa fue la campaña en cada municipio. Lo más cercano es el **gasto de campaña que fiscaliza el INE**, que se publica por candidatura y **distrito federal**, solo para las diputaciones federales.

- **Fuente:** INE, "Rendición de cuentas y resultados de fiscalización", datos abiertos, saldos finales después de los ajustes de fiscalización.
- **Años:** 2018 y 2021, con la distritación de 2017. Para 2015 no hay datos por distrito.
- **Verificado por separado:** el archivo de 2021 coincide con el dictamen consolidado INE/CG1413/2021 (ingresos idénticos; gastos con una diferencia de $45,491 que corresponde a los prorrateos que el dictamen descuenta). El total de 2018 no se pudo cotejar con el dictamen.

**Qué preguntamos:** en los distritos donde se gastó más por elector en 2021, ¿la participación salió más alta de lo que el modelo esperaba? Si las campañas explicaran el error que se agrupa por distrito (sección 6), debería haber una relación clara.

**Limitaciones:** es gasto en diputaciones federales, no en las campañas locales (gubernatura, ayuntamientos), y se conoce **después** de la elección, así que no sirve para predecir 2027. Incluye prorrateos de gastos compartidos. Otras fuentes que revisamos: la publicidad política en redes de Meta solo da rangos de gasto y alcance por estado, y requiere una cuenta con token; la de Google existe para México solo desde 2024; la encuesta de uso de internet y redes del INEGI (ENDUTIH) solo llega a nivel estatal.

In [12]:
import re
import unicodedata
FISCALIZACION = {2018: 'data/raw/externos/fiscalizacion/Saldos_Finales_FEDERAL_180828.csv', 2021: 'data/raw/externos/fiscalizacion/PREFO_20-21_CAM_SF_311022.csv'}
ENTIDADES = {'AGUASCALIENTES': 1, 'BAJA CALIFORNIA': 2, 'BAJA CALIFORNIA SUR': 3, 'CAMPECHE': 4, 'COAHUILA': 5, 'COLIMA': 6, 'CHIAPAS': 7, 'CHIHUAHUA': 8,
             'CIUDAD DE MEXICO': 9, 'DURANGO': 10, 'GUANAJUATO': 11, 'GUERRERO': 12, 'HIDALGO': 13, 'JALISCO': 14, 'MEXICO': 15, 'ESTADO DE MEXICO': 15,
             'MICHOACAN': 16, 'MORELOS': 17, 'NAYARIT': 18, 'NUEVO LEON': 19, 'OAXACA': 20, 'PUEBLA': 21, 'QUERETARO': 22, 'QUINTANA ROO': 23,
             'SAN LUIS POTOSI': 24, 'SINALOA': 25, 'SONORA': 26, 'TABASCO': 27, 'TAMAULIPAS': 28, 'TLAXCALA': 29, 'VERACRUZ': 30, 'YUCATAN': 31, 'ZACATECAS': 32}
def normalizar(s):
    s = unicodedata.normalize('NFKD', s.strip().upper())
    return ''.join(c for c in s if not unicodedata.combining(c))
def pesos(s):
    s = s.strip()
    if s in ('', '-', '$-', '$ -'):
        return 0.0
    v = float(re.sub(r'[^0-9.]', '', s) or 0)
    return -v if (s.startswith('-') or s.startswith('(') or '-$' in s) else v
gasto = []
for anio, ruta in FISCALIZACION.items():
    with (raiz / ruta).open('rb') as f:
        assert hashlib.file_digest(f, 'sha256').hexdigest() == huellas[ruta], f'{ruta} no coincide con su huella'
    filas_csv = list(csv.reader(io.StringIO((raiz / ruta).read_bytes().decode('cp1252'))))
    h = {normalizar(x): i for i, x in enumerate(filas_csv[0]) if x.strip()}
    entidad_col = h.get('ESTADO ELECCION', h.get('ENTIDAD FEDERATIVA'))
    for r in filas_csv[1:]:
        cargo = normalizar(r[h['CARGO']]) if len(r) > h['GASTOS REPORTADOS'] else ''
        m = re.match(r'\s*DISTRITO\s+(\d+)', r[h['SUBNIVEL ENTIDAD']], re.I) if cargo else None
        if 'DIPUTA' in cargo and cargo.endswith('MR') and m:
            gasto.append({'anio': anio, 'EDOCVE': ENTIDADES[normalizar(r[entidad_col])], 'DEF': int(m.group(1)), 'gasto': pesos(r[h['GASTOS REPORTADOS']])})
gasto = pl.DataFrame(gasto).group_by('anio', 'EDOCVE', 'DEF').agg(pl.col('gasto').sum(), pl.len().alias('contabilidades'))
totales = gasto.group_by('anio').agg(pl.len().alias('distritos'), pl.col('gasto').sum().round(2).alias('gasto_reportado')).sort('anio')
print(totales)
assert totales['distritos'].to_list() == [300, 300]
assert totales['gasto_reportado'].to_list() == [682_509_816.64, 1_281_256_101.68], 'No coincide con los totales verificados'

# Error del modelo (aprende de 2015, predice 2021) promediado por distrito, ponderado por la lista de cada sección
lista_2021 = pl.scan_parquet(P / 'ccpc/ccpc-2021.parquet').group_by(pl.col('EDOCVE').cast(pl.Int64), 'SECCION', 'DEF').agg(pl.col('LN').sum()).collect()
por_seccion = (secciones.filter(pl.col('AELEC') == 2021).with_columns(pl.col('EDOCVE').cast(pl.Int64)).join(lista_2021, on=['EDOCVE', 'SECCION'])
               .join(A21.select(pl.col('cvegeo').alias('municipio_2024')).with_columns(error=pl.Series(error_2021)), on='municipio_2024', how='left')
               .filter(pl.col('error').is_not_null()))
por_distrito = por_seccion.group_by('EDOCVE', 'DEF').agg(lista=pl.col('LN').sum(), error=(pl.col('error') * pl.col('LN')).sum() / pl.col('LN').sum())
filas = []
for anio in (2021, 2018):
    x = por_distrito.join(gasto.filter(pl.col('anio') == anio).with_columns(pl.col('EDOCVE').cast(pl.Int64), pl.col('DEF').cast(por_distrito['DEF'].dtype)), on=['EDOCVE', 'DEF'])
    x = x.with_columns(gasto_por_elector=pl.col('gasto') / pl.col('lista'))
    r, bajo, alto = spearman_ic(x['gasto_por_elector'].to_numpy(), x['error'].to_numpy(), veces=1000)
    filas.append({'Gasto de campaña de': anio, 'Distritos': x.height, 'Gasto mediano por elector ($)': round(x['gasto_por_elector'].median(), 1),
                  'ρ con el error del modelo en 2021 [IC 95 %]': f'{r:.2f} [{bajo:.2f}, {alto:.2f}]'})
pl.DataFrame(filas)

shape: (2, 3)
┌──────┬───────────┬─────────────────┐
│ anio ┆ distritos ┆ gasto_reportado │
│ ---  ┆ ---       ┆ ---             │
│ i64  ┆ u32       ┆ f64             │
╞══════╪═══════════╪═════════════════╡
│ 2018 ┆ 300       ┆ 6.8251e8        │
│ 2021 ┆ 300       ┆ 1.2813e9        │
└──────┴───────────┴─────────────────┘


Gasto de campaña de,Distritos,Gasto mediano por elector ($),ρ con el error del modelo en 2021 [IC 95 %]
i64,i64,f64,str
2021,300,13.3,"""0.05 [-0.05, 0.16]"""
2018,300,7.2,"""-0.15 [-0.26, -0.04]"""


**Qué vemos:**

- **El gasto de campaña de 2021 no explica el error del modelo por distrito:** la correlación es cercana a cero y su intervalo incluye el cero. Los distritos donde se gastó más no votaron más de lo que el modelo esperaba.
- **El gasto de 2018 sí tiene una relación débil y negativa** con el error de 2021. Como se conoce antes de la elección, en principio podría servir para predecir. Pero no hay datos de 2012 para probarlo con 2015, así que no cumple nuestra regla de validar en las dos transiciones. **Es una pista, no un hallazgo.**
- **Conclusión:** la parte del error que se agrupa por estado y distrito (sección 6) no se explica por el gasto en diputaciones federales. Probablemente responde a la coyuntura local de cada elección (campañas a gubernatura y ayuntamientos, conflictos, candidaturas). Eso cambia cada vez y no se puede anticipar con datos públicos previos.

## 10 · ¿Las variables también funcionan en las elecciones presidenciales?

Hasta aquí todo se probó en las dos transiciones de presidencial a intermedia (2015 y 2021). Agregamos las tres transiciones de intermedia a presidencial: **2009 → 2012, 2015 → 2018 y 2021 → 2024**. Así cada variable se prueba en **5 elecciones** en vez de 2.

**Mismo protocolo:** el residuo de la presidencial dada la intermedia anterior, contra cada variable, con información disponible antes de cada elección.

| Presidencial | Parte de | Datos sociales | Censo completo (ITER) | Migración (CONAPO) | Homicidios |
|---|---|---|---|---|---|
| 2012 | intermedia 2009 | censo 2010 | 2010 | 2010 | 2009–2011 |
| 2018 | intermedia 2015 | Intercensal 2015 | 2010 | 2010 | 2015–2017 |
| 2024 | intermedia 2021 | censo 2020 | 2020 | 2020 | 2021–2023 |

La Intercensal 2015 no trae migración a otro país ni a otra entidad por separado; esas dos variables se prueban solo en 2012 y 2024. El crecimiento de la lista no existe para 2012, porque no tenemos la elección de 2006.

**Cómo leer el resultado:**

- **"Estable en las 5":** mismo signo y distinto de cero en todas. Es un rasgo estructural del municipio.
- **"Estable, con signo opuesto":** estable en las dos intermedias y en las tres presidenciales, pero con signos contrarios. No es una contradicción: mide **qué tan sensible es el municipio al ciclo** presidencial–intermedia.
- **"Solo en intermedias"**, **"solo en presidenciales"** o **"no estable"**.

In [13]:
panel = pl.read_parquet(P / 'panel_municipal.parquet')
concurrencia = pl.read_parquet(P / 'concurrencia_municipal.parquet')
lista_edad = pl.read_parquet(P / 'lista_edad_sexo_municipal.parquet')
proyecciones = pl.read_parquet(E / 'conapo_proyecciones_edad_sexo.parquet')
poblacion_anual = proyecciones.group_by('cvegeo', 'anio').agg(poblacion=pl.col('poblacion').sum())
homicidios_mun = pl.read_parquet(E / 'inegi_homicidios_municipio.parquet').filter(pl.col('clave_presente_en_edr_anio'))
cobertura = (lista_edad.filter(pl.col('grupo_edad') != '18-19')
             .join(proyecciones.rename({'anio': 'AELEC'}).with_columns(pl.col('AELEC').cast(pl.Int16)), on=['cvegeo', 'AELEC', 'sexo', 'grupo_edad'])
             .group_by('cvegeo', 'AELEC').agg(cobertura_lista_20mas=pl.col('LN').sum() / pl.col('poblacion').sum() * 100))
casillas_inicio = {a: electores_por_casilla(a).with_columns(pl.when(pl.col('electores_por_casilla').is_finite()).then(pl.col('electores_por_casilla')).alias('electores_por_casilla'))
                   for a in (2009, 2015, 2021)}

PRESIDENCIALES = {  # elección: (intermedia de la que parte, presidencial anterior, datos sociales, ITER, migración, homicidios)
    2012: (2009, None, 2010, 2010, 2010, (2009, 2011)),
    2018: (2015, 2012, 2015, 2010, 2010, (2015, 2017)),
    2024: (2021, 2018, 2020, 2020, 2020, (2021, 2023)),
}
def tabla_presidencial(T):
    Q, previa, C, C_iter, C_migr, (a0, a1) = PRESIDENCIALES[T]
    social = [pl.col('cvegeo'), pl.col('grado_promedio_escolaridad').alias('escolaridad_promedio'), pl.col('pct_3ymas_habla_lengua_indigena').alias('pct_habla_lengua_indigena'),
              pl.col('pct_viviendas_con_internet').alias('pct_viviendas_internet'), pl.col('poblacion_total').log().alias('poblacion_log')]
    if C != 2015:
        social += [pl.col('pct_5ymas_residia_otro_pais_hace5').alias('pct_residia_otro_pais_hace5'), pl.col('pct_5ymas_residia_otra_entidad_hace5').alias('pct_residia_otra_entidad_hace5')]
    t = panel.filter(pl.col('AELEC') == Q).select('cvegeo', desviacion_base=pl.col('desviacion'), lista_base=pl.col('LN'),
        pct_rural=pl.col('LN_rural') / pl.col('LN') * 100, pct_hombres=pl.col('LN_hombres') / pl.col('LN') * 100,
        pct_20_29=pl.col('LN_20_29') / pl.col('LN') * 100, pct_65_mas=pl.col('LN_65_mas') / pl.col('LN') * 100)
    uniones = [
        panel.filter(pl.col('AELEC') == T).select('cvegeo', desviacion_objetivo=pl.col('desviacion'), sin_casilla_objetivo=pl.col('sin_casilla')),
        concurrencia.filter(pl.col('AELEC') == T).select('cvegeo', ayuntamiento_mismo_dia=pl.col('ayuntamiento_mismo_dia').cast(pl.Float64),
                                                         gobernador_mismo_dia=pl.col('gobernador_mismo_dia').cast(pl.Float64),
                                                         diputacion_local_mismo_dia=pl.col('diputacion_local_mismo_dia').cast(pl.Float64), es_sni=pl.col('es_sni').cast(pl.Float64)),
        cobertura.filter(pl.col('AELEC') == Q).select('cvegeo', 'cobertura_lista_20mas'),
        (homicidios_mun.filter(pl.col('anio').is_between(a0, a1)).group_by('cvegeo').agg(h=pl.col('homicidios').sum(), n=pl.len()).filter(pl.col('n') == a1 - a0 + 1)
         .join(poblacion_anual.filter(pl.col('anio') == (a0 + a1) // 2), on='cvegeo').select('cvegeo', homicidios_100mil_log=(pl.col('h') / (a1 - a0 + 1) / pl.col('poblacion') * 1e5).log1p())),
        pl.read_parquet(E / f'inegi_municipal_{C}.parquet').select(social),
        pl.read_parquet(E / 'coneval_pobreza_municipal.parquet').filter(pl.col('anio') == C).select(
            'cvegeo', pct_pobreza='pobreza_pct', pct_pobreza_extrema='pobreza_extrema_pct', pct_carencia_salud='carencia_salud_pct', pct_rezago_educativo='carencia_rezago_educativo_pct'),
        pl.read_parquet(E / 'conapo_marginacion.parquet').filter(pl.col('anio') == C).select(
            'cvegeo', marginacion=-pl.col('indice_marginacion'), pct_localidades_menos_5000='pct_poblacion_localidades_menos_5000'),
        pl.read_parquet(E / 'conapo_intensidad_migratoria.parquet').filter(pl.col('anio') == C_migr).select(
            'cvegeo', pct_viviendas_emigrantes='pct_viviendas_emigrantes_eua', pct_viviendas_remesas='pct_viviendas_reciben_remesas', pct_viviendas_retorno='pct_viviendas_migrantes_retorno'),
        censos[C_iter], casillas_inicio[Q], area,
    ]
    if previa is not None:
        uniones.append(panel.filter(pl.col('AELEC') == previa).select('cvegeo', lista_previa=pl.col('LN')))
    for u in uniones:
        t = t.join(u, on='cvegeo', how='left')
    crecimiento = (pl.col('lista_base') / pl.col('lista_previa') - 1) * 100 if previa is not None else pl.lit(None, dtype=pl.Float64)
    return t.with_columns(crecimiento_lista=crecimiento, densidad_log=(pl.col('poblacion_censo') / pl.col('km2')).log())

def intermedia(T):
    return (ampliada.filter(pl.col('eleccion_objetivo') == T)
            .rename({'desviacion_presidencial': 'desviacion_base', 'ayuntamiento_mismo_dia_intermedia': 'ayuntamiento_mismo_dia',
                     'gobernador_mismo_dia_intermedia': 'gobernador_mismo_dia', 'diputacion_local_mismo_dia_intermedia': 'diputacion_local_mismo_dia'}))

VARIABLES_CINCO = (['ayuntamiento_mismo_dia', 'gobernador_mismo_dia', 'diputacion_local_mismo_dia', 'es_sni', 'pct_rural', 'pct_hombres', 'pct_20_29', 'pct_65_mas',
                    'crecimiento_lista', 'cobertura_lista_20mas', 'escolaridad_promedio', 'pct_habla_lengua_indigena', 'pct_viviendas_internet',
                    'pct_residia_otro_pais_hace5', 'pct_residia_otra_entidad_hace5', 'poblacion_log', 'pct_pobreza', 'pct_pobreza_extrema', 'pct_carencia_salud',
                    'pct_rezago_educativo', 'marginacion', 'pct_localidades_menos_5000', 'pct_viviendas_emigrantes', 'pct_viviendas_remesas', 'pct_viviendas_retorno',
                    'homicidios_100mil_log'] + NUEVAS)
filas = []
for T, d, tipo in [(2015, intermedia(2015), 'intermedia'), (2021, intermedia(2021), 'intermedia')] + [(T, tabla_presidencial(T), 'presidencial') for T in PRESIDENCIALES]:
    d = d.filter(pl.col('desviacion_objetivo').is_not_null() & pl.col('desviacion_base').is_not_null() & ~pl.col('sin_casilla_objetivo').fill_null(False))
    pendiente, ordenada = np.polyfit(d['desviacion_base'], d['desviacion_objetivo'], 1)
    residuo = d['desviacion_objetivo'].to_numpy() - (ordenada + pendiente * d['desviacion_base'].to_numpy())
    for v in VARIABLES_CINCO:
        x = d[v].cast(pl.Float64).to_numpy() if v in d.columns else np.full(d.height, np.nan)
        if (~np.isnan(x)).sum() < 100:
            continue
        r, bajo, alto = spearman_ic(x, residuo, veces=300)
        filas.append({'variable': v, 'eleccion': str(T), 'tipo': tipo, 'rho': r, 'signo': 1 if bajo > 0 else (-1 if alto < 0 else 0)})
cinco = pl.DataFrame(filas)
lectura = (cinco.group_by('variable').agg(
        *[pl.col('rho').filter(pl.col('eleccion') == e).first().round(2).alias(e) for e in ['2015', '2021', '2012', '2018', '2024']],
        int_pos=(pl.col('signo').filter(pl.col('tipo') == 'intermedia') == 1).all(), int_neg=(pl.col('signo').filter(pl.col('tipo') == 'intermedia') == -1).all(),
        pre_pos=(pl.col('signo').filter(pl.col('tipo') == 'presidencial') == 1).all(), pre_neg=(pl.col('signo').filter(pl.col('tipo') == 'presidencial') == -1).all())
    .with_columns(lectura=pl.when((pl.col('int_pos') & pl.col('pre_pos')) | (pl.col('int_neg') & pl.col('pre_neg'))).then(pl.lit('1. estable en las 5'))
                  .when((pl.col('int_pos') | pl.col('int_neg')) & (pl.col('pre_pos') | pl.col('pre_neg'))).then(pl.lit('2. estable, con signo opuesto'))
                  .when(pl.col('int_pos') | pl.col('int_neg')).then(pl.lit('3. solo en intermedias'))
                  .when(pl.col('pre_pos') | pl.col('pre_neg')).then(pl.lit('4. solo en presidenciales')).otherwise(pl.lit('5. no estable')))
    .sort('lectura', 'variable'))
print(lectura.group_by('lectura').len().sort('lectura'))
lectura.select('variable', pl.col('2015').alias('Intermedia 2015'), pl.col('2021').alias('Intermedia 2021'), pl.col('2012').alias('Presidencial 2012'),
               pl.col('2018').alias('Presidencial 2018'), pl.col('2024').alias('Presidencial 2024'), pl.col('lectura').alias('Lectura'))

shape: (5, 2)
┌───────────────────────────────┬─────┐
│ lectura                       ┆ len │
│ ---                           ┆ --- │
│ str                           ┆ u32 │
╞═══════════════════════════════╪═════╡
│ 1. estable en las 5           ┆ 2   │
│ 2. estable, con signo opuesto ┆ 10  │
│ 3. solo en intermedias        ┆ 19  │
│ 4. solo en presidenciales     ┆ 4   │
│ 5. no estable                 ┆ 9   │
└───────────────────────────────┴─────┘


variable,Intermedia 2015,Intermedia 2021,Presidencial 2012,Presidencial 2018,Presidencial 2024,Lectura
str,f64,f64,f64,f64,f64,str
"""ayuntamiento_mismo_dia""",0.66,0.44,0.08,0.21,0.37,"""1. estable en las 5"""
"""es_sni""",-0.08,-0.34,-0.24,-0.26,-0.16,"""1. estable en las 5"""
"""cobertura_lista_20mas""",0.29,0.2,-0.31,-0.1,-0.11,"""2. estable, con signo opuesto"""
"""crecimiento_lista""",-0.19,-0.16,null,0.07,0.05,"""2. estable, con signo opuesto"""
"""densidad_log""",-0.3,-0.19,0.19,0.12,0.17,"""2. estable, con signo opuesto"""
"""electores_por_casilla""",-0.18,-0.14,0.05,0.06,0.21,"""2. estable, con signo opuesto"""
"""homicidios_100mil_log""",0.12,0.05,-0.22,-0.18,-0.11,"""2. estable, con signo opuesto"""
"""pct_65_mas""",0.23,0.13,-0.17,-0.12,-0.12,"""2. estable, con signo opuesto"""
"""pct_catolica""",0.22,0.08,-0.23,-0.17,-0.04,"""2. estable, con signo opuesto"""


**Qué vemos:**

- **Solo dos variables son estables en las 5 elecciones, con el mismo signo:** elegir ayuntamiento el mismo día y usos y costumbres. Son los rasgos estructurales del municipio, y son justo las que más pesan en el modelo.
- **Diez variables son estables, pero con signo opuesto entre intermedias y presidenciales:** densidad, % de 65 años o más, % católicos, viviendas deshabitadas, homicidios, cobertura de la lista, electores por casilla, crecimiento de la lista, discapacidad y % sin religión.
  - Miden **qué tan sensible es el municipio al ciclo**. Los municipios densos y urbanos **suben más en las presidenciales y bajan más en las intermedias**. Los rurales, con más personas mayores y más católicos, votan más parejo entre una y otra.
  - Para 2027, que es una intermedia, la dirección que importa es la de las intermedias, validada dos veces. **Esta prueba no las invalida: confirma el mecanismo.**
- **19 variables solo funcionan en intermedias** (por ejemplo, escolaridad, % de hombres y gobernador el mismo día). Siguen siendo útiles para 2027, pero con menos respaldo que las anteriores.
- **4 solo funcionan en presidenciales:** migración a otro país, emigrantes, remesas y retorno. Explica por qué la migración no sirvió para predecir intermedias (notebook 03b).
- **9 no son estables** en ningún sentido, entre ellas la pobreza, la desocupación y el automóvil.

**Consecuencia para el modelo:** confirma lo que ya usa. Las dos variables estructurales pesan más, y las demás aportan por su relación con el ciclo. No hay que quitar ni agregar variables.

## 11 · Conclusión y decisiones

**Lo que podemos decir ante el jurado:**

- Probamos 18 variables más de otras fuentes, con hipótesis escritas antes, 14 tipos de modelo y las variables en cinco elecciones (dos intermedias y tres presidenciales). El modelo **no quedó corto de información**: lo que anticipa la abstención en una intermedia es la historia del municipio y qué se elige ese día.
- La parte que no se puede anticipar es la coyuntura de cada estado y distrito en cada elección. El gasto de campaña federal no la explica.
- La revisión de los casos atípicos encontró un error real en el calendario (7 municipios con sistemas normativos indígenas fuera de Oaxaca), que ya se corrigió.

**Decisiones de Sofía Ordaz, 20 de octubre de 2026:**

| # | Decisión | Qué se adoptó y por qué |
|---|---|---|
| **D20** | ¿Agregar la densidad, los ocupantes por vivienda o el % de católicos? | **No.** Mejoran menos de 1 % y el semáforo casi no cambiaría |
| **D21** | ¿Cambiar el modelo de Huber al ensamble Huber + random forest? | **Sí.** Gana con la regla de selección fijada antes de ver resultados y baja el error en las dos validaciones (−0.17 y −0.41 pp). Huber sigue sirviendo para explicar qué pesa. Aplicado en el notebook 04 |

**Fuentes que no se pueden usar para predecir 2027:**

- **Consulta popular de 2021, revocación de mandato de 2022 y elección judicial de 2025** (INE). Miden la participación sin elección local, pero no hay nada parecido antes de 2015 o 2021 para validarlas. Podrían ir al mapa como capa de contexto.
- **Violencia contra autoridades y candidaturas** (bases de organizaciones civiles). Empiezan en 2018, así que solo alcanzan para una validación.
- **Gasto de campaña de 2027.** Se conoce después de la elección.